# Ordered probit

## Model

The latent response and threshold rule are

\[
y_n^*=\beta_1x_{n1}+\beta_2x_{n2}+\varepsilon_n,\qquad
y_n=q\ \text{if}\ \tau_{q-1}<y_n^*\leq\tau_q.
\]

For \(\varepsilon_n\sim\mathcal N(0,1)\),

\[
\Pr(y_n=q)=
\Phi(\tau_q-\boldsymbol x_n^\mathsf T\boldsymbol\beta)
-\Phi(\tau_{q-1}-\boldsymbol x_n^\mathsf T\boldsymbol\beta).
\]

The example estimates two covariate effects and three thresholds from data
generated by an ordered-probit process.

This notebook is self-contained and was executed in the repository's Office
validation environment. Install the released package with
`pip install torchdcm`, then select CPU or CUDA through the `device` argument.


In [1]:
import numpy as np
import torch

from IPython.display import HTML, display

from torchdcm import Beta, OrderedChoiceDataset, OrderedProbit
torch.set_default_dtype(torch.float64)
torch.manual_seed(7)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"TorchDCM example running on {device}")


TorchDCM example running on cuda


In [2]:
rng = np.random.default_rng(19)
n_obs = 800
x_1 = rng.normal(size=n_obs)
x_2 = rng.normal(size=n_obs)
latent_utility = 0.90 * x_1 - 0.60 * x_2 + rng.normal(size=n_obs)
y = np.digitize(latent_utility, bins=[-1.20, -0.20, 0.90])

data = OrderedChoiceDataset(
    y=torch.as_tensor(y, dtype=torch.long),
    x={
        "x_1": torch.as_tensor(x_1),
        "x_2": torch.as_tensor(x_2),
    },
    weights=torch.ones(n_obs),
    categories=[0, 1, 2, 3],
)
print("Category counts:", dict(zip(*np.unique(y, return_counts=True))))


Category counts: {np.int64(0): np.int64(156), np.int64(1): np.int64(197), np.int64(2): np.int64(250), np.int64(3): np.int64(197)}


In [3]:
latent = (
    Beta("B_X1", init=0.50) * "x_1"
    + Beta("B_X2", init=-0.30) * "x_2"
)
thresholds = [
    Beta("TAU_1", init=-1.00),
    Beta("TAU_2", init=-0.10),
    Beta("TAU_3", init=0.80),
]
model = OrderedProbit(latent, thresholds, device=device, max_iter=80)
result = model.fit(data)
display(HTML(result.report().to_html()))


Model family,OrderedProbit
Estimation objective,Maximum log likelihood
TorchDCM version,0.1.0
PyTorch version,2.12.1+cu130
Python version,3.12.13
Operating system,Linux-6.17.0-35-generic-x86_64-with-glibc2.39
Device,cuda
Tensor dtype,float64
Optimizer,torch.optim.LBFGS
Maximum iterations,80
Line search,strong_wolfe
